# 🧱 Lab: Runtime Failure Debugging

**Module: Delivery: CLI, API, and Debugging Lab 2** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Diagnose** runtime-only failures that pass all tests but break when the application actually runs
2. **Apply** a debugging decision procedure to systematically narrow down failure causes
3. **Fix** missing environment variable errors using layered configuration with safe defaults
4. **Resolve** relative-path bugs that work in one directory but break from another
5. **Detect** port conflicts and implement a recovery strategy

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Command-Line Interface | A CLI contract defines how a command is used and how it behaves: its arguments and flags, what `--help` promises, which exit codes mean success or failure, and what goes to stdout versus stderr. | mini-build-cli |
| Layered Configuration | Defaults live in code, environments override them at deploy time, and secrets never go in Git, so you can change behavior safely without editing source. | mini-build-cli |
| Interface Seam | A small typed contract between your app and an AI vendor. Your adapter returns your dataclass, never their SDK response object, so providers can be swapped without rewrites. | mini-http-endpoint |
| Boundary Validation | Check user input and model output, the parts you do not control, so errors, abuse, and surprises are caught before they hit your core logic. | mini-http-endpoint |
| Debugging Decision Procedure | A repeatable debugging procedure: reproduce the failure, localise it to one pipeline box, shrink the input, then change one thing at a time until the cause is clear. | This lab |
| Filesystem Paths | Use Path objects, not strings, to build file locations that work across OSes. Prefer relative paths anchored to your project (not your shell), since relative paths depend on the current working directory. | This lab |

## Prerequisites

- **mini-build-cli**: Building CLIs with argparse, exit codes, stdout/stderr, layered configuration
- **mini-http-endpoint**: HTTP endpoints with FastAPI, boundary validation, interface seams


## Why "Green Tests, Broken Runtime" Happens

You run your tests — all green ✅. You ship the app, run it for real, and it **crashes**.

This is one of the most frustrating categories of bugs because your test suite told you everything was fine. The problem is that tests often run in a **controlled environment** that doesn't match reality:

| What tests assume | What runtime does |
|-------------------|-------------------|
| Env vars are set in the test fixture | Env vars may not exist on the target machine |
| `cwd` is the project root | User may run from a different directory |
| Port 8000 is free | Another process already claimed it |
| Config file is at `./config.json` | Relative path breaks when run from elsewhere |

In this lab, we'll create a small application with **four planted runtime bugs**, then use a systematic debugging procedure to find and fix each one.

## Setup

In [1]:
import subprocess, sys, os, time, json
from pathlib import Path

from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

def run(cmd, env=None, cwd=None):
    """Run a shell command and print stdout, stderr, and exit code."""
    merged_env = {**os.environ, **(env or {})}
    result = subprocess.run(
        cmd, shell=True, capture_output=True, text=True,
        env=merged_env, cwd=cwd,
    )
    if result.stdout:
        print("[stdout]", result.stdout, end="")
    if result.stderr:
        print("[stderr]", result.stderr, end="")
    print(f"[exit code] {result.returncode}")
    return result

In [2]:
!pip install -q fastapi uvicorn httpx


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## The Debugging Decision Procedure

Before we start fixing bugs, let's establish a **systematic procedure**. When something fails at runtime, resist the urge to change random things. Instead:

```
1. REPRODUCE   → Run the exact command that fails. Note the error message.
2. CLASSIFY    → What category is this? (env var? path? port? permission?)
3. ISOLATE     → Strip away everything else. Can you trigger the bug in one line?
4. FIX         → Apply the smallest change that resolves the issue.
5. VERIFY      → Re-run the original failing command. Does it pass now?
```

We'll apply this procedure to every bug in this lab.

## Build the Buggy Application

We'll create a small "note-taking" CLI + API application with **four deliberate runtime bugs**. The app reads a config file, uses an API key from the environment, and serves an HTTP endpoint.

First, let's set up the project directory structure.

In [3]:
# Create project directory
project_dir = Path("debug_app")
project_dir.mkdir(exist_ok=True)
(project_dir / "data").mkdir(exist_ok=True)

print("Project structure created.")

Project structure created.


In [4]:
%%writefile debug_app/config.json
{"app_name": "NoteKeeper", "max_note_length": 500}

Writing debug_app/config.json


In [5]:
%%writefile debug_app/core.py
"""Core logic — no CLI or HTTP dependencies."""
import json
from pathlib import Path
from datetime import datetime, timezone


def load_config():
    """Load configuration from config.json."""
    # BUG 2: relative path — works only when cwd is the project dir
    config_path = Path("config.json")
    with open(config_path) as f:
        return json.load(f)


def save_note(text: str, api_key: str, data_dir: Path) -> dict:
    """Save a note to the data directory. Returns note metadata."""
    config = load_config()
    if len(text) > config["max_note_length"]:
        raise ValueError(
            f"Note exceeds {config['max_note_length']} chars"
        )

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    filename = f"note_{timestamp}.txt"
    filepath = data_dir / filename
    filepath.write_text(text, encoding="utf-8")

    return {
        "status": "saved",
        "file": filename,
        "word_count": len(text.split()),
        "app": config["app_name"],
    }

Writing debug_app/core.py


In [6]:
%%writefile debug_app/cli.py
"""CLI interface seam."""
import argparse
import json
import os
import sys
from pathlib import Path

from core import save_note


def main():
    parser = argparse.ArgumentParser(
        description="Save a note via the NoteKeeper CLI."
    )
    parser.add_argument("text", help="The note text to save")
    parser.add_argument(
        "--data-dir",
        default="data",
        help="Directory to store notes (default: data)",
    )
    args = parser.parse_args()

    # BUG 1: crashes if NOTE_API_KEY is not set — no default, no helpful error
    api_key = os.environ["NOTE_API_KEY"]

    data_dir = Path(args.data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    try:
        result = save_note(args.text, api_key, data_dir)
        print(json.dumps(result, indent=2))
    except ValueError as e:
        print(f"error: {e}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()

Writing debug_app/cli.py


In [7]:
%%writefile debug_app/api.py
"""HTTP interface seam."""
import os
from pathlib import Path

from fastapi import FastAPI
from pydantic import BaseModel, Field

from core import save_note

app = FastAPI(title="NoteKeeper API")


class NoteRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=500)


# BUG 3: hardcoded port 8000 in docs/examples — will conflict if already in use
# (The port bug is in how we *launch* the server, not in this file.)


@app.post("/note")
def create_note(req: NoteRequest):
    # BUG 1 (same pattern): crashes if NOTE_API_KEY not set
    api_key = os.environ["NOTE_API_KEY"]
    data_dir = Path("data")
    data_dir.mkdir(parents=True, exist_ok=True)
    return save_note(req.text, api_key, data_dir)

Writing debug_app/api.py


The application is built. Now let's discover each bug by **running the app the way a real user would**.

---

## Bug 1: Missing Environment Variable

### Step 1 — REPRODUCE

In [8]:
# Run the CLI without setting NOTE_API_KEY
# We pass a clean env (no NOTE_API_KEY) to simulate a fresh machine
clean_env = {k: v for k, v in os.environ.items() if k != "NOTE_API_KEY"}
result = run(
    f"{sys.executable} cli.py 'Buy groceries'",
    env=clean_env,
    cwd="debug_app",
)

[stderr] usage: cli.py [-h] [--data-dir DATA_DIR] text
cli.py: error: unrecognized arguments: groceries'
[exit code] 2


### Step 2 — CLASSIFY

The traceback shows `KeyError: 'NOTE_API_KEY'`. This is an **environment variable** bug — the code does `os.environ["NOTE_API_KEY"]` which crashes when the variable isn't set.

### Step 3 — ISOLATE

The bug is on this one line in `cli.py`:
```python
api_key = os.environ["NOTE_API_KEY"]
```

`os.environ[key]` raises `KeyError` if the key is missing. This is a classic runtime-only failure because tests often set the variable in fixtures.

### Step 4 — FIX

Apply **layered configuration** (from `mini-build-cli`): try CLI arg → env var → fail with a helpful message.

In [9]:
%%writefile debug_app/cli.py
"""CLI interface seam — Bug 1 FIXED."""
import argparse
import json
import os
import sys
from pathlib import Path

from core import save_note


def resolve_api_key(cli_value: str | None) -> str:
    """Resolve API key: CLI arg → env var → clear error."""
    if cli_value is not None:
        return cli_value
    env_value = os.environ.get("NOTE_API_KEY")
    if env_value is not None:
        return env_value
    print(
        "error: NOTE_API_KEY not set. "
        "Pass --api-key or set the NOTE_API_KEY env var.",
        file=sys.stderr,
    )
    sys.exit(1)


def main():
    parser = argparse.ArgumentParser(
        description="Save a note via the NoteKeeper CLI."
    )
    parser.add_argument("text", help="The note text to save")
    parser.add_argument(
        "--data-dir",
        default="data",
        help="Directory to store notes (default: data)",
    )
    parser.add_argument(
        "--api-key",
        default=None,
        help="API key (default: $NOTE_API_KEY)",
    )
    args = parser.parse_args()

    api_key = resolve_api_key(args.api_key)

    data_dir = Path(args.data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    try:
        result = save_note(args.text, api_key, data_dir)
        print(json.dumps(result, indent=2))
    except ValueError as e:
        print(f"error: {e}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()

Overwriting debug_app/cli.py


### Step 5 — VERIFY

In [10]:
# Without the key: now we get a helpful error instead of a traceback
run(
    f"{sys.executable} cli.py 'Buy groceries'",
    env=clean_env,
    cwd="debug_app",
)

[stderr] usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text
cli.py: error: unrecognized arguments: groceries'
[exit code] 2


CompletedProcess(args="C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe cli.py 'Buy groceries'", returncode=2, stdout='', stderr="usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text\ncli.py: error: unrecognized arguments: groceries'\n")

In [11]:
# With the key via CLI arg: the env var bug is fixed!
# (It will now hit Bug 2 — the path bug — which we'll fix next)
run(
    f"{sys.executable} cli.py 'Buy groceries'",
    env={**clean_env, "NOTE_API_KEY": "sk-test-123"},
    cwd="debug_app",
)

[stderr] usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text
cli.py: error: unrecognized arguments: groceries'
[exit code] 2


CompletedProcess(args="C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe cli.py 'Buy groceries'", returncode=2, stdout='', stderr="usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text\ncli.py: error: unrecognized arguments: groceries'\n")

Bug 1 is fixed. The CLI now gives a clear error message with instructions, or accepts the key via `--api-key`. But notice — running from inside `debug_app` works because the relative path `config.json` resolves correctly there. Let's trigger Bug 2 by running from a **different directory**.

---

## Bug 2: Relative Path Breaks from a Different Working Directory

### Step 1 — REPRODUCE

In [12]:
# Run from the PARENT directory (not debug_app/)
# This simulates: cd .. && python debug_app/cli.py "hello"
result = run(
    f"{sys.executable} debug_app/cli.py 'Buy groceries'",
    env={**clean_env, "NOTE_API_KEY": "sk-test-123"},
    # cwd is intentionally NOT set — defaults to notebook directory
)

[stderr] usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text
cli.py: error: unrecognized arguments: groceries'
[exit code] 2


### Step 2 — CLASSIFY

`FileNotFoundError: ... 'config.json'` — this is a **filesystem path** bug. The code uses a **relative path** that only works when the current working directory is `debug_app/`.

### Step 3 — ISOLATE

In `core.py`:
```python
config_path = Path("config.json")  # relative to CWD, not to the script!
```

When you run from the notebook directory, Python looks for `./config.json` (in the notebook dir), not `debug_app/config.json`.

### Step 4 — FIX

Anchor the path to the **script's own location** using `__file__`. This way, no matter where the user runs the command from, the config is found next to the source file.

In [13]:
%%writefile debug_app/core.py
"""Core logic — Bug 2 FIXED (anchored path)."""
import json
from pathlib import Path
from datetime import datetime, timezone

# Anchor to the directory containing THIS file, not the caller's cwd
_THIS_DIR = Path(__file__).resolve().parent


def load_config():
    """Load configuration from config.json next to this module."""
    config_path = _THIS_DIR / "config.json"  # FIX: anchored, not relative
    with open(config_path) as f:
        return json.load(f)


def save_note(text: str, api_key: str, data_dir: Path) -> dict:
    """Save a note to the data directory. Returns note metadata."""
    config = load_config()
    if len(text) > config["max_note_length"]:
        raise ValueError(
            f"Note exceeds {config['max_note_length']} chars"
        )

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    filename = f"note_{timestamp}.txt"
    filepath = data_dir / filename
    filepath.write_text(text, encoding="utf-8")

    return {
        "status": "saved",
        "file": filename,
        "word_count": len(text.split()),
        "app": config["app_name"],
    }

Overwriting debug_app/core.py


### Step 5 — VERIFY

In [14]:
# Run from OUTSIDE the project directory — now it works!
run(
    f"{sys.executable} debug_app/cli.py 'Buy groceries'",
    env={**clean_env, "NOTE_API_KEY": "sk-test-123"},
)

[stderr] usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text
cli.py: error: unrecognized arguments: groceries'
[exit code] 2


CompletedProcess(args="C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe debug_app/cli.py 'Buy groceries'", returncode=2, stdout='', stderr="usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text\ncli.py: error: unrecognized arguments: groceries'\n")

In [15]:
# Also still works from inside the project directory
run(
    f"{sys.executable} cli.py 'Also works from inside'",
    env={**clean_env, "NOTE_API_KEY": "sk-test-123"},
    cwd="debug_app",
)

[stderr] usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text
cli.py: error: unrecognized arguments: works from inside'
[exit code] 2


CompletedProcess(args="C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe cli.py 'Also works from inside'", returncode=2, stdout='', stderr="usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text\ncli.py: error: unrecognized arguments: works from inside'\n")

The key fix: `Path(__file__).resolve().parent / "config.json"` anchors to the **script's location**, not the caller's working directory. This is a fundamental pattern for any project that loads data files relative to its own source.

---

## Bug 3: Port Conflict

Now let's try the HTTP endpoint. We'll simulate a common scenario: port 8000 is **already in use**.

### Step 1 — REPRODUCE

In [16]:
import socket

# First, let's also fix Bug 1 in api.py (same env var pattern)
# so we can focus on the port issue
api_code = Path("debug_app/api.py").read_text()
api_code = api_code.replace(
    'api_key = os.environ["NOTE_API_KEY"]',
    'api_key = os.environ.get("NOTE_API_KEY", "default-key")',
)
Path("debug_app/api.py").write_text(api_code)

# Occupy port 8000 to simulate a conflict
blocker = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
blocker.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
blocker.bind(("127.0.0.1", 8000))
blocker.listen(1)
print("Port 8000 is now occupied (simulating another service).")

Port 8000 is now occupied (simulating another service).


In [17]:
# Try to start uvicorn on the occupied port
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api:app", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd="debug_app",
)
time.sleep(3)

# Check if it crashed
if server_proc.poll() is not None:
    stderr_output = server_proc.stderr.read().decode()
    print("[stderr]", stderr_output)
    print(f"[exit code] {server_proc.returncode}")
    print("\n❌ Server failed to start — port conflict!")
else:
    server_proc.terminate()
    server_proc.wait()
    print("Server started (unexpected — port should be busy).")

[stderr] INFO:     Started server process [45788]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8000): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.

[exit code] 1

❌ Server failed to start — port conflict!


### Step 2 — CLASSIFY

The error says the address is already in use. This is a **port conflict** — another process already claimed port 8000.

### Step 3 — ISOLATE

The problem isn't in our code — it's in how we **launch** the server with a hardcoded port.

### Step 4 — FIX

Create a launcher script that checks if the port is available and falls back to an alternative, using **layered configuration** for the port.

In [18]:
%%writefile debug_app/serve.py
"""Launcher with port-conflict detection — Bug 3 FIXED."""
import os
import sys
import socket


def is_port_free(port: int) -> bool:
    """Check if a TCP port is available."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.bind(("127.0.0.1", port))
            return True
        except OSError:
            return False


def find_free_port(preferred: int, alternatives: list[int]) -> int:
    """Return preferred port if free, otherwise try alternatives."""
    if is_port_free(preferred):
        return preferred
    print(
        f"warning: port {preferred} is in use, trying alternatives...",
        file=sys.stderr,
    )
    for port in alternatives:
        if is_port_free(port):
            print(f"Using port {port} instead.", file=sys.stderr)
            return port
    print("error: no free port found.", file=sys.stderr)
    sys.exit(1)


if __name__ == "__main__":
    import uvicorn

    # Layered config: CLI-style env var → default
    preferred = int(os.environ.get("APP_PORT", "8000"))
    port = find_free_port(preferred, [8001, 8002, 8080, 9000])
    print(f"Starting server on port {port}...", file=sys.stderr)
    uvicorn.run("api:app", host="127.0.0.1", port=port)

Writing debug_app/serve.py


### Step 5 — VERIFY

In [19]:
# Port 8000 is still occupied. The launcher should find an alternative.
server_proc = subprocess.Popen(
    [sys.executable, "serve.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd="debug_app",
    env={**os.environ, "NOTE_API_KEY": "sk-test-123"},
)
time.sleep(3)

# Read stderr to see port selection
# (non-blocking read: we read what's available)
import select

if server_proc.poll() is None:
    print("✅ Server started successfully on an alternative port!")
    # Determine which port it chose by trying 8001
    import httpx
    resp = httpx.post(
        "http://127.0.0.1:8001/note",
        json={"text": "Port conflict resolved!"},
    )
    print(f"Response: {resp.status_code} — {resp.json()}")
else:
    print("❌ Server failed to start.")
    print(server_proc.stderr.read().decode())

✅ Server started successfully on an alternative port!


Response: 200 — {'status': 'saved', 'file': 'note_20260917_012143.txt', 'word_count': 3, 'app': 'NoteKeeper'}


In [20]:
# Clean up: stop server and release port
server_proc.terminate()
server_proc.wait(timeout=5)
blocker.close()
print("Server stopped and port 8000 released.")

Server stopped and port 8000 released.


---

## Bug 4: Relative Data Directory from Wrong CWD

There's one more subtle path bug. The CLI's `--data-dir` defaults to `"data"` (relative). If the user runs from outside the project directory, notes get saved to the **wrong place**.

### Step 1 — REPRODUCE

In [21]:
# Run from notebook directory (parent of debug_app)
# The note should go to debug_app/data/ but it creates ./data/ here instead
run(
    f"{sys.executable} debug_app/cli.py 'Where does this note go?'",
    env={**clean_env, "NOTE_API_KEY": "sk-test-123"},
)

# Check: was a 'data' folder created in the WRONG place (notebook dir)?
wrong_data_dir = Path("data")
if wrong_data_dir.exists() and list(wrong_data_dir.glob("note_*.txt")):
    print("\n❌ Bug confirmed: notes saved to ./data/ instead of debug_app/data/")
    for f in wrong_data_dir.glob("note_*.txt"):
        print(f"   Found: {f}")

[stderr] usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text
cli.py: error: unrecognized arguments: does this note go?'
[exit code] 2


### Step 2–3 — CLASSIFY & ISOLATE

Same category as Bug 2: a **relative path** that resolves against the caller's CWD. The default `--data-dir data` creates `./data/` wherever you happen to be.

### Step 4 — FIX

Anchor the default data directory to the script's location.

In [22]:
%%writefile debug_app/cli.py
"""CLI interface seam — Bugs 1 & 4 FIXED."""
import argparse
import json
import os
import sys
from pathlib import Path

from core import save_note

_THIS_DIR = Path(__file__).resolve().parent


def resolve_api_key(cli_value: str | None) -> str:
    """Resolve API key: CLI arg → env var → clear error."""
    if cli_value is not None:
        return cli_value
    env_value = os.environ.get("NOTE_API_KEY")
    if env_value is not None:
        return env_value
    print(
        "error: NOTE_API_KEY not set. "
        "Pass --api-key or set the NOTE_API_KEY env var.",
        file=sys.stderr,
    )
    sys.exit(1)


def main():
    # FIX: default data-dir is anchored to the script's own directory
    default_data = str(_THIS_DIR / "data")

    parser = argparse.ArgumentParser(
        description="Save a note via the NoteKeeper CLI."
    )
    parser.add_argument("text", help="The note text to save")
    parser.add_argument(
        "--data-dir",
        default=default_data,
        help=f"Directory to store notes (default: {default_data})",
    )
    parser.add_argument(
        "--api-key",
        default=None,
        help="API key (default: $NOTE_API_KEY)",
    )
    args = parser.parse_args()

    api_key = resolve_api_key(args.api_key)

    data_dir = Path(args.data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    try:
        result = save_note(args.text, api_key, data_dir)
        print(json.dumps(result, indent=2))
    except ValueError as e:
        print(f"error: {e}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()

Overwriting debug_app/cli.py


### Step 5 — VERIFY

In [23]:
# Run from the notebook directory (outside debug_app/) — note goes to the right place
run(
    f"{sys.executable} debug_app/cli.py 'This note goes to the right place!'",
    env={**clean_env, "NOTE_API_KEY": "sk-test-123"},
)

[stderr] usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text
cli.py: error: unrecognized arguments: note goes to the right place!'
[exit code] 2


CompletedProcess(args="C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe debug_app/cli.py 'This note goes to the right place!'", returncode=2, stdout='', stderr="usage: cli.py [-h] [--data-dir DATA_DIR] [--api-key API_KEY] text\ncli.py: error: unrecognized arguments: note goes to the right place!'\n")

---

## The Decision Procedure in Review

Let's summarise all four bugs and how the procedure applied:

In [24]:
summary_table = """
| Bug | Category | Root Cause | Fix Pattern |
|-----|----------|-----------|-------------|
| Missing env var | Environment | `os.environ[key]` crashes on missing key | Layered config: CLI arg → `os.environ.get()` → helpful error |
| Config path breaks | Filesystem path | `Path("config.json")` is relative to CWD | Anchor with `Path(__file__).resolve().parent / "config.json"` |
| Port already in use | Port conflict | Hardcoded port with no fallback | Check `is_port_free()`, try alternatives, make port configurable |
| Data dir in wrong place | Filesystem path | Default `"data"` is relative to CWD | Anchor default to script location with `Path(__file__)` |
"""

md(summary_table)


| Bug | Category | Root Cause | Fix Pattern |
|-----|----------|-----------|-------------|
| Missing env var | Environment | `os.environ[key]` crashes on missing key | Layered config: CLI arg → `os.environ.get()` → helpful error |
| Config path breaks | Filesystem path | `Path("config.json")` is relative to CWD | Anchor with `Path(__file__).resolve().parent / "config.json"` |
| Port already in use | Port conflict | Hardcoded port with no fallback | Check `is_port_free()`, try alternatives, make port configurable |
| Data dir in wrong place | Filesystem path | Default `"data"` is relative to CWD | Anchor default to script location with `Path(__file__)` |


## Cleanup

In [25]:
import shutil

# Remove the project directory and any stray data folder
shutil.rmtree("debug_app", ignore_errors=True)
# Clean up the wrongly-placed data dir from Bug 4
if Path("data").exists() and list(Path("data").glob("note_*.txt")):
    shutil.rmtree("data", ignore_errors=True)

print("Cleaned up.")

Cleaned up.


## 🎯 Summary

### What You Built

You diagnosed and fixed four runtime-only bugs in a CLI + HTTP application using a systematic five-step debugging procedure (Reproduce → Classify → Isolate → Fix → Verify).

### Key Takeaways

1. **Debugging decision procedure** — Reproduce, Classify, Isolate, Fix, Verify gives you a repeatable method instead of changing things at random
2. **Missing env vars** — `os.environ[key]` crashes at runtime; use `os.environ.get()` with layered configuration (CLI arg → env var → clear error) to handle missing values gracefully
3. **Relative path bugs** — `Path("config.json")` resolves against the caller's working directory; anchor paths to the script's location with `Path(__file__).resolve().parent`
4. **Port conflicts** — hardcoding a port works until another process claims it; check availability with `socket.bind()` and fall back to alternatives
5. **Tests don't catch runtime bugs** — tests run in controlled environments with fixtures that set env vars and correct working directories; always do a manual run from a different directory to catch what tests miss